# Consultas Analíticas

En esta sección se realizan consultas SQL sobre el modelo dimensional construido en la capa Gold.

Las consultas permiten obtener indicadores académicos y financieros que servirán como base para la creación de dashboards.

In [0]:
fact = spark.table(f"rendimiento_estudiantil.gold.fact_matriculas")

dim_estudiantes = spark.table(f"rendimiento_estudiantil.gold.dim_estudiantes")

dim_cursos = spark.table(f"rendimiento_estudiantil.gold.dim_cursos")

dim_semestres = spark.table(f"rendimiento_estudiantil.gold.dim_semestres")

Registrar vistas sql

In [0]:
fact.createOrReplaceTempView("fact_matriculas")

dim_estudiantes.createOrReplaceTempView("dim_estudiantes")

dim_cursos.createOrReplaceTempView("dim_cursos")

dim_semestres.createOrReplaceTempView("dim_semestres")

#Rendimiento Académico

KP1 1: Promedio General

In [0]:
%sql
SELECT
ROUND(AVG(nota_final),2) AS Promedio_General
FROM fact_matriculas;

Promedio de notas por curso

In [0]:
%sql
SELECT

c.nombre_curso,

ROUND(AVG(f.nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_cursos c

ON f.id_curso=c.id_curso

GROUP BY c.nombre_curso

ORDER BY promedio;

Riesgo por estado académico

In [0]:
%sql
SELECT

estado,

COUNT(*) cantidad

FROM fact_matriculas

GROUP BY estado;

Riesgo según modalidad

In [0]:
%sql
SELECT

modalidad,

COUNT(*) estudiantes

FROM fact_matriculas f

JOIN dim_semestres s

ON f.Semester_ID=s.Semester_ID

GROUP BY modalidad;

Nota promedio según modalidad

In [0]:
%sql
SELECT

modalidad,

ROUND(AVG(nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_semestres s

ON f.Semester_ID=s.Semester_ID

GROUP BY modalidad;

Edad vs Rendimiento

In [0]:
%sql
SELECT

Age,

ROUND(AVG(nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY Age

ORDER BY Age;

Rendimiento por género

In [0]:
%sql
SELECT
    TRIM(Gender) AS Gender,
    ROUND(AVG(nota_final),2) AS promedio
FROM fact_matriculas f
JOIN dim_estudiantes e
    ON f.Student_ID = e.Student_ID
GROUP BY TRIM(Gender);

Rendimiento por nivel socioeconómico

In [0]:
%sql
SELECT

Family_Income_Level,

ROUND(AVG(nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY Family_Income_Level

ORDER BY Family_Income_Level;

Rendimiento por nivel educativo de los padres

In [0]:
%sql
SELECT

Parent_Education,

ROUND(AVG(nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY Parent_Education;

Riesgo por carrera

In [0]:
%sql
SELECT

Major_Subject,

ROUND(AVG(nota_final),2) promedio,

COUNT(*) estudiantes

FROM fact_matriculas f

JOIN dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY Major_Subject;

Evolución del rendimiento por semestre

In [0]:
%sql
SELECT

periodo,

ROUND(AVG(nota_final),2) promedio

FROM fact_matriculas f

JOIN dim_semestres s

ON f.Semester_ID=s.Semester_ID

GROUP BY periodo

ORDER BY periodo;

# Vistas para Power Bi

Vista 1
Rendimiento de cada estudiante

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_rendimiento_estudiantes AS

SELECT

f.Student_ID,

e.Gender,

e.Age,

e.Major_Subject,

e.Region_Type,

AVG(f.nota_final) AS Promedio_Nota,

COUNT(f.id_matricula) AS Cursos_Llevados,

SUM(f.costo_matricula) AS Inversion,

MAX(f.estado) AS Estado

FROM rendimiento_estudiantil.gold.fact_matriculas f

JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY

f.Student_ID,

e.Gender,

e.Age,

e.Major_Subject,

e.Region_Type;

Vista 2
Factores familiares

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_factores_familiares AS

SELECT

Parent_Education,

Family_Income_Level,

COUNT(*) Total_Estudiantes,

AVG(Age) Edad_Promedio

FROM rendimiento_estudiantil.gold.dim_estudiantes

GROUP BY

Parent_Education,

Family_Income_Level;

Vista 3
Rendimiento por carrera

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_rendimiento_carreras AS

SELECT

e.Major_Subject,

COUNT(*) Total,

ROUND(AVG(f.nota_final),2) Promedio

FROM rendimiento_estudiantil.gold.fact_matriculas f

JOIN rendimiento_estudiantil.gold.dim_estudiantes e

ON f.Student_ID=e.Student_ID

GROUP BY

e.Major_Subject;

Vista 4
Cursos críticos

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_cursos_criticos AS

SELECT

c.nombre_curso,

AVG(f.nota_final) Promedio,

COUNT(*) Matriculados

FROM rendimiento_estudiantil.gold.fact_matriculas f

JOIN rendimiento_estudiantil.gold.dim_cursos c

ON f.id_curso=c.id_curso

GROUP BY

c.nombre_curso;

Vista 5
Evolución por semestre

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_semestres AS

SELECT

s.periodo,

AVG(f.nota_final) Promedio,

COUNT(*) Matriculas

FROM rendimiento_estudiantil.gold.fact_matriculas f

JOIN rendimiento_estudiantil.gold.dim_semestres s

ON f.Semester_ID=s.Semester_ID

GROUP BY

s.periodo

ORDER BY

s.periodo;

Vista 6
Riesgo académico

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_riesgo_estudiantes AS

SELECT

Student_ID,

AVG(nota_final) promedio,

CASE

WHEN AVG(nota_final)<60 THEN 'Alto Riesgo'

WHEN AVG(nota_final)<75 THEN 'Riesgo Medio'

ELSE 'Bajo Riesgo'

END Nivel_Riesgo

FROM rendimiento_estudiantil.gold.fact_matriculas

GROUP BY Student_ID;

Vista 7
Indicadores generales

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_indicadores AS

SELECT

COUNT(DISTINCT Student_ID) Estudiantes,

COUNT(id_matricula) Matriculas,

ROUND(AVG(nota_final),2) Promedio,

ROUND(SUM(costo_matricula),2) Ingreso

FROM rendimiento_estudiantil.gold.fact_matriculas;

Vista 8

Dashboard principal

In [0]:
%sql
CREATE OR REPLACE VIEW rendimiento_estudiantil.gold.vw_dashboard AS

WITH rendimiento_estudiante AS (

    SELECT
        Student_ID,
        ROUND(AVG(nota_final),2) AS Promedio_Nota
    FROM rendimiento_estudiantil.gold.fact_matriculas
    GROUP BY Student_ID

)

SELECT

    f.id_matricula,

    f.Student_ID,

    e.Gender,

    e.Age,

    e.Region_Type,

    e.Family_Income_Level,

    e.Parent_Education,

    e.Major_Subject,

    c.id_curso,

    c.nombre_curso,

    c.nivel,

    s.Semester_ID,

    s.periodo,

    s.anio,

    s.ciclo,

    s.modalidad,

    f.fecha_matricula,

    f.nota_final,

    r.Promedio_Nota,

    CASE
        WHEN r.Promedio_Nota < 60 THEN 'Alto'
        WHEN r.Promedio_Nota < 75 THEN 'Medio'
        ELSE 'Bajo'
    END AS Nivel_Riesgo,

    f.estado,

    f.costo_matricula

FROM rendimiento_estudiantil.gold.fact_matriculas f

INNER JOIN rendimiento_estudiantil.gold.dim_estudiantes e
    ON f.Student_ID = e.Student_ID

INNER JOIN rendimiento_estudiantil.gold.dim_cursos c
    ON f.id_curso = c.id_curso

INNER JOIN rendimiento_estudiantil.gold.dim_semestres s
    ON f.Semester_ID = s.Semester_ID

INNER JOIN rendimiento_estudiante r
    ON f.Student_ID = r.Student_ID;